# Machine Learning

## Text Processing
## Data Preparation
## Classification - Binary
## Clustering

# 1 Text Processing

In [ ]:
spark.stop()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
        .appName('ProgBigData-Spark - wk5 ML Demo') \
        .master('local[*]') \
        .getOrCreate()

In [ ]:
sFile = 'datasets/shakespeare.txt'
sDF = spark.read.text(sFile)
sDF.show(10,truncate=False)

In [ ]:
# Split input into Tokens (individual words)
wordsDF = sDF.select(explode(split(col("value"), " ")).alias("word"))
wordsDF.show(20, truncate=False)

In [ ]:
# Remove lines that don't have any Tokens
wordsDF = wordsDF.select(col("word")).where(col("word") != "")
wordsDF.show(20, truncate=False)

In [ ]:
# Do a word count
wordsDF.groupBy("word").count().orderBy(col("count").desc()).show(10)

In [ ]:
# Consolidate the above examples by making allwords lower
# and not including lines that don't have tokens

wordsDF2 = wordsDF.select(lower(col("word")).alias("word")).where(col("word") != "")
wordsDF2.groupBy("word").count().orderBy(col("count").desc()).show(10)

In [ ]:
# PySpark Built-in Text Processing
from pyspark.ml.feature import Tokenizer, RegexTokenizer

shake_data = spark.read.text(sFile)
shake_data.show(10, truncate=False)

# Tokenize the data
tokenizer = Tokenizer(inputCol='value', outputCol='Tokens')
tnk = tokenizer.transform(shake_data)
tnk.show(20)

# 2 Classification - Data Processing

In [ ]:
# Define the schema for the adult.csv
adult_schema = 'age DOUBLE, workclass STRING, fnlwgt DOUBLE, education STRING, education_num DOUBLE, marital_status STRING, occupation STRING, relationship STRING, race STRING, sex STRING, capital_gain DOUBLE, capital_loss DOUBLE, hours_per_week DOUBLE, native_country STRING, income STRING'

# Read into a DF
file_path = 'datasets/adult.csv'
dataset = spark.read.csv(file_path, schema=adult_schema)
dataset.show(5, vertical=True)

In [ ]:
# Create training and test data sets and set seed for reproducibility
# Training DF = 70%
# Test DF = 30%
trainDF, testDF = dataset.randomSplit([0.7,0.3], seed=42)

print('Full data set size = ', dataset.count())
print('Training set size = ', trainDF.cache().count(), '(', trainDF.count()/dataset.count(), '%)') # Cache because accessing training data multiple times
print('Test set size = ', testDF.count(), '(', testDF.cache().count()/dataset.count(), '%)')

In [ ]:
# trainDF.summary().show(vertical=True)
# trainDF.describe().show()
# trainDF.printSchema()
trainDF.select("income").show(5)

In [ ]:
# Distribution of the number of hours_per_week
trainDF.select("hours_per_week").summary().show()

In [ ]:
# Education status
trainDF.groupBy("education").count().sort("count", ascending=False).show()

In [ ]:
# Important Concepts in MLlib Machine Learning 

## TRANSFORMER: Takes a DF as input, and returns a new DF. 
## Transformers do not learn any parameters from the data and simply apply rule-based
## transformations to eitehr prepare data for model training or generate predictions using
## a trained MLlib model. Called with .transform() 

## ESTIMATOR: Learns or 'fits' parameters from your DF via a .fit() method
## and returns a Model, which is a transformer.

## PIPELINE: Combines multiple steps into a single workflow that can be easily run.
## Creating a ML model typically invovles setting up many different steps and iterating
## over them. Pipelines help automate this process.

## https://spark.apache.org/docs/latest/ml-pipeline.html#ml-pipelines

### Question/Problem: Build a model that predicts the income level
### from the features included in the dataset 
### (education level, marital status, occupation, and so on).

In [ ]:
# StringIndexer and OneHotEncoder, used to convert categorical variables
# into a set of numeric variables that only take on values 0 and 1.

# STRINGINDEXER: Converts a column of string values to a column of label indexes
# e.g., "red", "blue", and "green" to 0, 1, and 2.

# ONEHOTENCODER: Maps a column of category indices to a column of binary vectors
# with at most one "1" in each row that indicates the category index for that row.
# Involves a two step process, stringindexer -> onehotencoder

from pyspark.ml.feature import StringIndexer, OneHotEncoder

categoricalCols = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex"]

# The ff two lines are estimators. Return funcions that we'll later apply to transform the dataset.
stringIndexer = StringIndexer(inputCols=categoricalCols, outputCols=[x + "Index" for x in categoricalCols], handleInvalid="keep")
encoder = OneHotEncoder(inputCols=stringIndexer.getOutputCols(), outputCols=[x + "OHE" for x in categoricalCols])

# The label column 'income' is also a string value, has two possible values "<=50K" and ">50K".
# Convert it to a numeric value
labelToIndex = StringIndexer(inputCol="income", outputCol="label")

In [ ]:
# Transformers accept a DF as input and returns a new DF with one/more columns appended
# They do not learn from your data but apply rule-based transformations using the transform() method.

stringIndexerModel = stringIndexer.fit(trainDF)
# stringIndexerModel.transform(trainDF).show()
stringIndexerModel.transform(trainDF).select("age","education","hours_per_week","income").show(truncate=False)

In [ ]:
# Combine all feature columns into a single feature vector.

# Most MLlib algorithms require a single features columns as input.
# Each row in this column contains a vector of data points corresponding 
# to the set of features used for prediction.

# MLlib provides the VectorAssembler transformer to create a single vector column froma list of columns.

from pyspark.ml.feature import VectorAssembler
# This includes both the numeric columns and the one-hot encoded binary vector columns in our dataset.
numericCols = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
assemblerInputs = [c + "OHE" for c in categoricalCols] + numericCols
vecAssembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")

# 3 Classification - Models

In [ ]:
# Define the Model - using logistic regression
from pyspark.ml.classification import LogisticRegression

# Define the model
lr = LogisticRegression(featuresCol="features", labelCol="label", regParam=1.0)

In [ ]:
# Build the pipeline
# A pipeline is an ordered list of transformers and estimators.
# You can define a pipeline to automate and ensure repeatability of the transformation to be applied to a dataset
# The pipeline.fit() method returns a PipelineModel, which is a transformer.

from pyspark.ml import Pipeline
from pyspark.sql.functions import col

# Because the dataset was loaded without "header=True", the actual header row became parsed as the first data row.
# This row contains NULLs for numeric columns, which causes the VectorAssembler to fail with handleInvalid="error"

# Fix this problem before fitting the pipeline
# by identifying the problematic role as 'age' being NULL.
filtered_trainDF = trainDF.filter(col("age").isNotNull())
filtered_testDF = testDF.filter(col("age").isNotNull())

# Define the pipeline based on stages created in prev steps
pipeline = Pipeline(stages=[stringIndexer, encoder, labelToIndex, vecAssembler, lr])

# Define the pipeline model
pipelineModel = pipeline.fit(filtered_trainDF)

# Apply the pipeline model to the test dataset
predDF = pipelineModel.transform(filtered_testDF)

In [ ]:
# Display the predictions from the model.
# The features column is a sparse vector, which is often the case after one-hot encoding, 
# because there are so many 0 values.
predDF.select("features", "label", "prediction", "probability").show()

In [ ]:
pipelineModel.stages[-1], predDF.drop("prediction", "rawPrediction", "probability").show()

In [ ]:
# display(pipelineModel.stages[-1], predDF.drop("prediction", "rawPrediction", "probability"), "ROC")
pipelineModel.stages[-1], predDF.select("rawPrediction").show(2, truncate=False)

In [ ]:
# Evaluate the model
# Use the BinaryClassificationEvaluator to evaluate the area under the ROC curve
# and MutliclassClassificationEvaluator to evaluate the accuracy

from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

bcEvaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")
print(f"Area under ROC curve: {bcEvaluator.evaluate(predDF)}")

mcEvaluator = MulticlassClassificationEvaluator(metricName="accuracy")
print(f"Accuracy: {mcEvaluator.evaluate(predDF)}")

In [ ]:
# SQL for additional analysis
predDF.createOrReplaceTempView("finalPredictions") # create TempView for SQL

sql = """select income, prediction from finalPredictions where prediction = '1'"""
# sql = """SELECT age, income, prediction FROM finalPredictions WHERE income LIKE '%>50K%' AND age >= 25"""
spark.sql(sql).show()

In [ ]:
sql = """SELECT age, prediction, count(*) AS count FROM finalPredictions GROUP BY age, prediction ORDER BY age"""
spark.sql(sql).show()

# 4 Clustering

In [ ]:
# Unsupervised Learning
# Dataset = Iris

iris_file = "datasets/iris.csv"
irisDF = spark.read.format("csv")\
  .option("header","true")\
  .option("inferSchema", "true")\
  .load(iris_file)

irisDF.show()

In [ ]:
# Define a schema with different labels
iris_schema = 'sepal_length DOUBLE, sepal_width DOUBLE, petal_length DOUBLE, petal_width DOUBLE, species STRING'
iris_file = "datasets/iris.csv"

irisDF2 = spark.read.format("csv").option("header","true")\
  .option("inferSchema", "false")\
  .schema(iris_schema)\
  .load(iris_file)

# Monotonically_increasing_id() starts at 0, so we add 1
# https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.monotonically_increasing_id.html
irisDF2 = irisDF2.withColumn("id", monotonically_increasing_id() + 1)

# Move ID to first position
irisDF2 = irisDF2.select("id", "sepal_length", "sepal_width", "petal_length", "petal_width", "species")

# display(irisDF2)
irisDF2.show()

In [ ]:
# Apply an Indexer to the categorical variable 'species'
indexer = StringIndexer(inputCol="species", outputCol="speciesIndex")

indexedDF = indexer.fit(irisDF2).transform(irisDF2)

indexedDF.show(10, truncate=False)

In [ ]:
# Prepare input dataset for Cluster Algorithm
from pyspark.ml.linalg import Vectors

def mapLibSVM(row):
    return (row[5],Vectors.dense(row[1:4]))

df = indexedDF.rdd.map(mapLibSVM).toDF(["label", "features"])
df.show()

In [ ]:
from pyspark.ml.feature import VectorAssembler

assemble=VectorAssembler(inputCols=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
 outputCol = 'iris_features')

assembled_data=assemble.transform(irisDF2)

In [ ]:
# Using the KMeans algorithm loop to find the (possible) best number of Clusters
# aka the Elbo Method
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

silhouette_scores=[]
evaluator = ClusteringEvaluator(featuresCol='iris_features', \
metricName='silhouette', distanceMeasure='squaredEuclidean')

for K in range(2,20):

    KMeans_=KMeans(featuresCol='iris_features', k=K)

    KMeans_fit=KMeans_.fit(assembled_data)

    KMeans_transform=KMeans_fit.transform(assembled_data)

    evaluation_score=evaluator.evaluate(KMeans_transform)

    silhouette_scores.append(evaluation_score)

# Display results
print(silhouette_scores)
type(silhouette_scores)

In [ ]:
# Plot the results
import numpy as np
import matplotlib.mlab as mlab
import matplotlib.pyplot as plt
import seaborn as sbs
from matplotlib.ticker import MaxNLocator

fig, ax = plt.subplots(1,1, figsize=(8,6))
ax.plot(range(2,20),silhouette_scores[0:18])
ax.set_xlabel('k')
ax.set_ylabel('cost')
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()

In [ ]:
# As there are 3 different values in 'Species' let's build for that
# e.g., 'Instead of looking at the species, group them by its dimensions'
KMeans_=KMeans(featuresCol='iris_features', k=3)
KMeans_Model=KMeans_.fit(assembled_data)
KMeans_Assignments=KMeans_Model.transform(assembled_data)


KMeans_Assignments.show()

In [ ]:
# To visualize the 4D data into 2, we'll use a dimensionality reduction technique called PCA.
# Spark has its own flavour of PCA.

# First, perofrm the PCA. k=2 represents the number of principal components
from pyspark.ml.feature import PCA as PCAml

pca = PCAml(k=2, inputCol='iris_features', outputCol='pca')
pca_model = pca.fit(assembled_data)
pca_transformed = pca_model.transform(assembled_data)

In [ ]:
# Next, extract the principal components
import numpy as np

x_pca = np.array(pca_transformed.rdd.map(lambda row: row.pca).collect())

# Retrieve the cluster assignments from k-means assignments
cluster_assignment = np.array(KMeans_Assignments.rdd.map(lambda row: row.prediction).collect()).reshape(-1,1)

In [ ]:
# Display the result
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pca_data = np.hstack((x_pca, cluster_assignment))

pca_df = pd.DataFrame(data=pca_data, columns=("1st_principal", "2nd_principal", "cluster_assignment"))
sns.FacetGrid(pca_df, hue="cluster_assignment", height=6).map(plt.scatter, "1st_principal", "2nd_principal").add_legend()

plt.show()